In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import folium
import io
import base64


In [ ]:
smesh = pd.read_csv("concatSmeshData/f864_airQualityMetrics.csv")
smesh.sort_values(by = "datetime", inplace = True, ignore_index = True)
smesh['datetime'] = pd.to_datetime(smesh['datetime'])
smesh['datetime'] = smesh['datetime'].dt.tz_localize("US/Pacific", ambiguous= True)


# smesh.set_index("datetime", inplace = True)
# smesh = smesh.groupby("fromNode")["pm25Environmental"].resample("20min").mean()

# smesh = smesh.reset_index()
smesh = smesh[["datetime","fromNode","pm25Environmental"]]

# smesh["datetime"] = smesh["datetimeCol"].iloc(1)

display(smesh.head())

In [ ]:


low = pd.Timestamp("2025-10-23-09", tz = "US/Pacific")
high = pd.Timestamp("2025-10-25-12", tz = "US/Pacific")

smeshLocations = pd.read_csv("Snode_Locations - SNode.csv")
purpleAirLocations = pd.read_csv("PurpleAir Download 11-7-2025/PurpleAirLocations.csv")
m = folium.Map(location=[38.570690,-122.687953], zoom_start=14,tiles="https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png",
    attr="Map data: © OpenStreetMap contributors, SRTM | Map style: © OpenTopoMap (CC-BY-SA)"
)

for node in smesh["fromNode"].unique():
    nodeName = node[-4:].upper()
    nodeData = smesh[smesh["fromNode"] == node]
    print(len(nodeData))
    print(nodeName)
    if len(nodeData) == 0:
        continue
    nodeData = nodeData[(nodeData["datetime"] >= low) & (nodeData["datetime"] <= high) ]
    fig, ax = plt.subplots()
    ax.plot(nodeData["datetime"], nodeData["pm25Environmental"])
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d %H", tz = "US/Pacific"))
    ax.tick_params(axis='x', rotation=45)
        
    plt.title(node)
    # plt.show()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches='tight')
    buf.seek(0)
    img_base64 = base64.b64encode(buf.read()).decode('utf-8')
    plt.close(fig)  # close figure to save memory

    # Create HTML popup with image
    html = f'<img src="data:image/png;base64,{img_base64}" width="400" height="300">'
    sensorIndex = smeshLocations[smeshLocations["SNODE"] == nodeName].index[0]
    # print(smeshLocations.iloc[i]["SNODE"])
    print(smeshLocations.loc[sensorIndex]["SNODE"])

    # folium.CircleMarker(location = [smeshLocations.iloc[i]["Lat"],smeshLocations.iloc[i]["Lon"]]).add_to(m)
    folium.CircleMarker(location = [smeshLocations.iloc[sensorIndex]["Lat"],smeshLocations.iloc[sensorIndex]["Lon"]], fill = True, popup=folium.Popup(html, max_width=500),tooltip=f"Node {node[-4:]} points").add_to(m)
#Fix for correct index
i = 0
for purpleAirName in purpleAirLocations["Sensor Name"].unique():

    folium.Circle(location = [purpleAirLocations.iloc[i]["Latitude"],purpleAirLocations.iloc[i]["Longitude"]],fill = True, tooltip=f"Node {purpleAirName}", color="purple").add_to(m)
    i+= 1

In [ ]:
alertCaLocations = pd.read_csv("Snode_Locations - AlertCALocations.csv")

for i in range(len(alertCaLocations)):
    print(alertCaLocations.iloc[i])
    folium.Circle(location = [alertCaLocations.iloc[i]["Lat"],purpleAirLocations.iloc[i]["Lon"]],fill = True, tooltip=f"AlertCA: {alertCaLocations.iloc[i]["Camera"]}", color="purple").add_to(m)

    # df.iloc

In [ ]:


low = pd.Timestamp("2025-10-23-09", tz = "US/Pacific")
high = pd.Timestamp("2025-10-25-12", tz = "US/Pacific")

smeshLocations = pd.read_csv("Snode_Locations - SNode.csv")

i = 0
for node in smesh["fromNode"].unique():
    nodeName = node[-4:].upper()
    nodeData = smesh[smesh["fromNode"] == node]
    print(len(nodeData))
    if len(nodeData) == 0:
        continue
    nodeData = nodeData[(nodeData["datetime"] >= low) & (nodeData["datetime"] <= high) ]
    fig, ax = plt.subplots()
    ax.plot(nodeData["datetime"], nodeData["pm25Environmental"])
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d %H", tz = "US/Pacific"))
    ax.tick_params(axis='x', rotation=45)
        
    plt.title(node)
    plt.show()